In [2]:
from diffrax import (
    AbstractSolver,
    AbstractStepSizeController,
    Heun,
    Tsit5,
    PIDController,
    diffeqsolve,
    ODETerm,
    Event,
    SaveAt
)
from jax import numpy as jnp

In [29]:
# pseudo-code (JAX/Diffrax style), matching your signature
def loss_y(y, target, lam): 
    return 0.5*(y - target)**2 + 0.5*lam*y**2

def neg_activity_grad(t, y, args):
    (target, lam, *_rest) = args
    g = (1 + lam) * y - target      # here 1+lam = 1.1, target = 2.0
    return -g


def steady_state_event_with_timeout(t, y, args, **kwargs):
    target, lam, tol, t_timeout, *_ = args
    grad = (1.0 + lam) * y - target
    grad_resid = jnp.abs(grad) - tol     # triggers when <= 0
    time_resid = (t_timeout - t)         # triggers when <= 0
    return jnp.minimum(grad_resid, time_resid)


# pack args to mirror your call
args = (2.0, 0.1, 1e-3, 10.0, None, None)   # target, lam, tol, timeout, ...
controller = PIDController(
    rtol=1e-4, atol=1e-6,
    dtmin=1e-6,           # avoid vanishing dt
    dtmax=0.25,           # <-- cap step growth to prevent 'inf' near steady state
    safety=0.9, factormin=0.2, factormax=2.0
)

solution = diffeqsolve(
    terms=ODETerm(neg_activity_grad),
    solver=Heun(),
    t0=0.0,
    t1=20.0,
    dt0=1e-2,
    y0=5.0,
    args=args,
    stepsize_controller=controller,
    event=Event(steady_state_event_with_timeout),
    saveat=SaveAt(t0=True, t1=True, steps=True),
)
y_star = solution.ys[-1]   # ≈ 1.8182


In [30]:
solution.ys

Array([5.       , 4.965193 , 4.9153323, ...,       inf,       inf,
             inf], dtype=float32)

In [28]:
sol.ys

Array([5.       , 4.965192 , 4.8967133, ...,       inf,       inf,
             inf], dtype=float32)